In [ ]:
# Install required libraries
!pip install -q bitsandbytes accelerate transformers


In [ ]:
import os
import sys
import json
import re
import time
from collections import Counter

print("=" * 70)
print("KAZAKH LLM BENCHMARK & LINGUISTIC DIAGNOSTIC ENGINE (GPU T4 x 2)")
print("Target: Qwen-2.5-7B-Instruct vs. Human KazSAnDRA (ACL/EMNLP Benchmark)")
print("=" * 70)

# 1. Install missing dependencies if needed
try:
    import bitsandbytes
except ImportError:
    print("Installing bitsandbytes for 4-bit LLM quantization...")
    os.system("pip install -q bitsandbytes accelerate")

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)")

# 2. Loanword lexicon
LOANWORD_SET = {
    "доставка", "заказ", "каспи", "возврат", "скидка", "курьер",
    "упаковка", "брак", "чек", "магазин", "товар", "качество",
    "приложение", "сервис", "бонус", "оплата", "размер", "цвет",
    "клиент", "сумма", "меню", "акция", "гарантия"
}

def tokenize_words(text: str):
    return re.findall(r"[a-zA-Zа-яА-ЯәіңғүұқөһӘІҢҒҮҰҚӨҺ]+", text.lower())

def calculate_token_inflation(text: str, tokenizer=None) -> float:
    words = text.strip().split()
    if not words:
        return 1.0
    if tokenizer is not None:
        try:
            tokens = tokenizer.encode(text, add_special_tokens=False)
            return round(len(tokens) / len(words), 3)
        except Exception:
            pass
    clean_words = tokenize_words(text)
    subword_count = sum(1 if len(w) <= 4 else (2 if len(w) <= 7 else (3 if len(w) <= 11 else 4)) for w in clean_words)
    return round(max(len(words), subword_count) / len(words), 3)

def calculate_lexical_diversity(texts: list) -> dict:
    all_words, bigrams = [], []
    for t in texts:
        w = tokenize_words(t)
        all_words.extend(w)
        if len(w) >= 2:
            for i in range(len(w) - 1):
                bigrams.append((w[i], w[i+1]))
    if not all_words:
        return {"ttr": 0.0, "distinct_1": 0.0, "distinct_2": 0.0, "total_words": 0}
    unique_words = len(set(all_words))
    return {
        "ttr": round(unique_words / len(all_words), 4),
        "distinct_1": round(unique_words / len(all_words), 4),
        "distinct_2": round(len(set(bigrams)) / max(1, len(bigrams)), 4) if bigrams else 0.0,
        "total_words": len(all_words),
        "unique_words": unique_words
    }

def detect_code_switched_loanwords(text: str) -> list:
    return [w for w in tokenize_words(text) if any(w.startswith(root) for root in LOANWORD_SET)]

def compute_profile(records: list, tokenizer=None) -> dict:
    grouped = {}
    for r in records:
        gen = r.get("generator", "unknown")
        grouped.setdefault(gen, []).append(r)

    summary = {}
    for gen, items in grouped.items():
        texts = [x["text"] for x in items]
        lex_div = calculate_lexical_diversity(texts)
        tw_vals = [calculate_token_inflation(t, tokenizer) for t in texts]
        loanword_counts = [len(detect_code_switched_loanwords(t)) for t in texts]
        
        by_bracket = {"short": [], "medium": [], "long": []}
        for x in items:
            b = x.get("length_bracket", "medium")
            if b in by_bracket:
                by_bracket[b].append(x)

        strat_tw = {}
        for b, b_items in by_bracket.items():
            strat_tw[b] = round(sum(calculate_token_inflation(x["text"], tokenizer) for x in b_items) / max(1, len(b_items)), 3) if b_items else 0.0

        summary[gen] = {
            "sample_count": len(items),
            "overall_ttr": lex_div["ttr"],
            "distinct_1": lex_div["distinct_1"],
            "distinct_2": lex_div["distinct_2"],
            "avg_tw": round(sum(tw_vals) / max(1, len(tw_vals)), 3),
            "stratified_tw": strat_tw,
            "loanword_frequency": sum(loanword_counts),
            "loanwords_per_100_words": round((sum(loanword_counts) / max(1, lex_div["total_words"])) * 100, 2)
        }
    return summary

def build_prompt(seed: dict) -> str:
    bracket = seed.get("length_bracket", "medium")
    domain = seed.get("domain", "consumer_reviews")
    if bracket == "short":
        len_instr = "Өте қысқа пікір жазыңыз (1-2 сөйлем, 10-15 сөзден аспасын)."
    elif bracket == "medium":
        len_instr = "Орташа ұзындықтағы шынайы пікір жазыңыз (2-3 сөйлем, 20-35 сөз)."
    else:
        len_instr = "Толыққанды, егжей-тегжейлі пікір жазыңыз (3-5 сөйлем, 40+ сөз)."

    return (
        "<|im_start|>system\n"
        "Сіз интернет-дүкендегі (Kaspi.kz) нақты сатып алушысыз. "
        "Қазақ тілінде шынайы пікір (review) жазыңыз. "
        "Ешқандай жасанды интеллект кіріспесіз немесе ескертусіз, қарапайым ауызекі халықтық тілде жазыңыз.<|im_end|>\n"
        f"<|im_start|>user\nТақырыбы: {domain}. {len_instr}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

def main():
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)

    # Self-contained embedded seed human reviews (1,000 samples from KazSAnDRA)
    import base64, zlib
    SEEDS_B64 = "eJztnd1uXMeRx+/9FMcCFtgFGGXOGc6Xb3yzD7A3e7VYBE5CxEYsC1CU3QUWCwxJxeGCSSTbyEQfselYXmCFAKuhKIrkDId8he7hG/gFwkdYVPUZkhqyRqenC405Z/4wElsUv87vdFfXv6q66l/eS5L/fC9JkuTW/bX/uH/rg+SW6Zn++Jldt9vJ+Jk5MUPz2vTNyOya/kpWS5vuj4nZNfu2a3tvfZLdNqeJeTN+abvm2PbMaMXsnj3hTzNvJt/2fOeL35/vPDy+teJ+8Kcf/XTt01sfJLX8z79Y+2zt3kf3796j3+bjX9/56LPJZ/7s44/u/eTTtc9+cf/jWx8kjebkG/BHfvLTex/97Jdr/Ay/+vjuvfuTr/r53TsfffIZffhndz/71a/vrN37yb21f/tk7d9/NfmMT35OP/69JPmvlRt4/Nn0x1/xM4+f2W1zYLfNseknZkAAzL4Z0H/27bo5NYeJXScayfil3aLnJ0x9u2437Lbtmtf0pQl99eQb7pqR6Zsj+psFw5EKOOo1eoBDM7KbdiMxg/FLu2F75jSxG+Ndfu3b9NgDc2JeJ+aN7bmFYo7NPj9nL/Q567rPmUmv/Y3pmwFvA7vJb0nhHaWp6u9el353eitb9F4uluz4WWJemUPz2hzydu3bbmIGZy/oxW3ZbV6PCg9Y133AVeEBz/7Xbph9sikn9JvbLr2lddM3p/QYph/8nlqqj9GQ3tOlJTh7wU80fmb6doPMRTJ+OVl1l6bm2OzbjSQ3oMPpfy/YzmpKT00msWuGblHy+zLHhOGqSZ0sSts93/lym/4X+nSZ7jttSU/3tdk3e7ZnDm48GwdnL2zXbpo9Wr67bivSO76dXD1ndqVX/8PX3wefE5kqh/a77ScdBu4ZFsx+diTz4jaj3aCVafp2S8Gi1HRP59rM5UeL68TQKjohq2837O8SWj+2Z4aTkzh4P+muo1RyOKZ2gNm1PdslX9K5lM/Z/G/YzXy78JFGZtS8cZ+/T++SH32Uu6H0Ts3odiiA1bYugKzEOykVXZGnF66xee2WpnMP++bIPnAGct/sBy9G5ceRHA/zjXM83lqT7p/gN7Kq+wiS00Enay5MVs6e8Om0f3EMOy9qdPXgGSTm1BnDQ9ulj5qhe4sfBm8g5ZcmOhyX8oyMud02w0Uz561K+0qp6CT8yTwyT8wX5pF5lpi/mD+a7xLzV/MX8515ZP5Mf0cf/dw8Nd+Zz8dfmm/Mk0V7tE41JEpWK3b8ToeEgh9DWc2LXsR3dpPeAcdnXtH+eWSO+GQdXTmYzCHFakzf7JGLkYd5bNfsmUMK3Djjt68RxVjV9Z4y0XkY5f4gu0QHHIPboseidfls0d6e6EV8y2/nDYfe3Ls6IofIjIKfoK77BKvLEjZrzNZO0177wG5yRGr7avjzMrB2vvPl4/OdL7853/nrt8FPqry1mu+w8FNPei26qxBWy5RXaasih1bbZxEumGTKOqV1V+u1MgfNRRfhMgBHHoLdcCv/4iVw2OCQwwz1xIzOXlCKi8Lou3SeUpSIdgmF2F8tmDKqZz4mbNHeV70Krk1djCg8vcjVTJ8k++xwhrs4unGqeuOdh8ckf0yJiUVbTdU7zestP1ds3265LTNiGXfkdM0xJ6yHtkcBH5UdVNcNaNXfddhfrrtpkXrza0yuBFe+4j+yOZ/+Yvqv853vXwTnXZRxVCTwsFrz2pDjr8wJLdnd8a4ZXlHuwXuyoftUy57NWM2WMue9+u6SDPqH8rh/O3y4YG796mqJ3cTVRqVD6KuS3+IegMsIyBBwZQ/HkijeybVli+ZArraW3TaWuWZi9R1FEwttJBqSs2G3x8/5VDoiQ3Gbsp0nbnWRt8GL7Mitwq9oc7kgAcVpuhyrobe2Ee5a1XXXWSOtgnJuZFWvcmzUS3zuNooENvJIBqUBelRkcHHaUrLjxG7kVdLbzvTlZjuvj9PJ8yjn2xqNyoURGs2lso2tAiHsDVf+MrGWuWnZNydkcDbdW/xwwRzFhuhcUCqRQj4Ts8/e1d5bkZNg29LRfZbOzGexvfFzshYDXm2bZkQa0hyze3j2guJZdPOEc6zuEkZeanv79sRXGbniTjolLrzF4DiPrihr1pYl0dqU3JW799eSX67dW7uzdn/BDr9mtoyHX7O+FAHn5rLUcjTF0MmJ/S0b2t/R05A7vW7Xuah8PzGntmsOzdCckPym0imFKl7dIGzTK9ezaKZFdFAekcrJr47k1zA1FpVyJVSzveRRnmanoh5ma/bdk+W5+tRKl/H0b2WVLeZu1ZdJArdWS1zF1WoUrgaY9lTswwWTua3q5HZareoah3Y1veSW5Kektdrf2XUXaOGoSm/BbEB7qbI57bTslyPa2XJezmnXl/S5V0ucKm6LQZHn7Flsu5Kl8XNu+UNv8f3gJ9CNX7WbSxGma7eqk/Bot5cl2N/uvFM7X/4T/JZ0UzIdr5LVBbNrnbS0TU86Wdk9oE69mhqiU+bK0U6pL7R0xK4XSxaN7bQqurcqGnfodMqd4EprS1OZkdYqUUma1rJlChilNdHbWLKjIa2JqQ666P05X/PmS957eYj9dNLfeJuhHATbTuVWV7XGUmj7tNZcui5mNa/rvAvmjaa1dlXuuae1d910Wa4Lx6ncc3RJO6TLLUurHddP5Valix+8S8VWpWWIVaRiZ9JlNUqNinl2chfTkpcNpmKT05J1rEjldqZLOjolFbugUkiADjVOl26bo8T+lps9TK7pkLWhOkdWmrbHOdXe+c7jz4MXr25iLs0q2aYklfuljlxSmzwR91Dri3aQyz1PS9WyNpW7npbgHYiBlcdmRO5tYp5y4d/35vvErp+9MAdk/cgIUgP4DbZ5XNJP5dEHTsO7TntH4a0P6sqGP6tQ0Wkq9jW9un3oH7txvvP48cKtvFZlimZTsY9pdVtSpnL701KlFErdCjWVe6FWpbdG6tf7tBSdGlK5JWopTLfYC3XhK87TerW7jKWe7VHLInPqVQk9iE1Qq5nBFpucVud8EvuflkD9yV1OSzuFTu5bWtJ4coGOpKXfQrMusuZ6fHF9cLFxaSnyUXJf0koeSGLv0nK8rPbSVdOslnfKSip2K12WFiqp3MG0pKex3My0ivFysbFpheOajfIGN8SmpiVLqImdTMvQXzsVO5OWOwQj9iUtcRxW7k9a0uNJbDZa7akdqdh6tBQ112IX0iWtCRT7ky7+ESz2HK2wxyS3Hy1D+EZsMlrpJoCp3J60XL6i3Ka0rEf4u+bCLfRRJvYWLcVBLDYErditPLEFaPVuPrfK60uI/T1LWCEnt/sshVVrLulFMbkdaBlcO7nj5zc3LDr3T/AzaJvqTrULltpLGrEQu4FWM+8qtw4tfV/hVG4PuvgVQGKLz7I08RHbfC5+O7xUbvBZsqpGuYVnpWMovt0+yyoXxV6fJU79dJbU7xDbh5bj6oHcQrQE89lTsX1oNZ0+satoCTs7p2KX0bJ4SnKj0RI0fZGbg37noif2gTlxVVZDZ70nu+TkxrypK4ekChIaIco+4sTc64aiGsoxjc6S+Bxid9EyJBIysbVoNaI2mdxPtCLPl1W8H0om9hatbCl5JjcRrciibSzJvNtM7Cq6RHPFM7HPaBVL0TOxL2nJYmWZ2JW0JDoik9uIlvXSaCY3Aq3KHctMbvlZ8raEWbp8nkxaocBKJncALYtBbJZ9rFEmtvas8B4q0LNrmbphZ2mpIy5if89KesNy28+qxCfE5qBlKBjM5JagS2pc5D6jlU63ZmLH0eoerGJf0go/cpm7imRi99LlvH6YiZ1NyxZqknublr7oMpvR+bRM99cyub3pwtcvZnIb02of6mL/0yrsq1IP083k/qeVKo/IxIaolcv1iR1Tq3ujPhPbppbCnxQ7olZXAMiNVJd0olAm92EtQQ+nTGy6evf+WvLLtXtrd9buL9zvXOYuqpnYRbU8dzUzsZfq4l8Ey97ZGLUsolfskVq53gaZ2Bu1bO9MbpRa9uIEsQFq6V5RVvIsvdjLtHp2QWxhuqQBXbkv6nJdHsnExqqls0at8t5lysROqqVw1MWeqeXrSpTJ7VLLIFKbvkUgfJjt8Rt6xeGFIw6AmgHpEi4McdrEjM53vv/r+c7jjUWrexE7pV5tNDAd4nV9BxZu7aFEZApIxa/niG1SS90XPJP7p5Y/GSY2Sa2eZhD7qFbwUTtlboGRiS1Wy+D7ih1WSxAiFbumVrLEWGycWrFMuthltQwDoDOxr2p5L+PJzVbLYZ5bJQ+Uyl1TKysrW9VrcJbJjVWr3bM4EzurlsKgi41Sy60S5R6p5VeJcg/VCjyb6F8851g1HQUb1L2OA1FkVt4P3lHKDl5VuqxmcpfVJRknmokNV523S6rqxhq6wdkL27Wbzh/OPTDu+JJcDbZeSXe+DfSHr78PDpZqH3JiDGPJUppiH9cS3J8ody/WrLNU8RCxd2t5TkKxH2vFQjpyq9bFDzKKXVqr9o6WqnvZO7u2LllFltjd1XzH1x0GtpfY39AbpaQvt+nLB/aRfeU/mkP7G5d94kVxfPaE1e/IHDpflqSxmyfQnXTpIs/vyCHm72xOXQNAt38YKW2gY74oxcwT+m7BbnBbthF31n7+ya/veOKry+1lH7k94C5a8mM6CMkP3a+S8VNqCZWvNPKLRhOi3BDRXdLscr8i5r7yFtbEvDl7QuW3CTchJ37H9FGyqPTN8qUYTKuVatOS3Kzx/7lZFM6SnriiCTeAwplTuia2Sc+auG6QVGbsLpe5Z6WU57rdNLt23XWEsw8Z+CaD4+lyw9wdn9zaMaMf5ZudXpEDx2UeJElGLEoWb7lJjt6Y3rpzezi0TU8w4Nt4Zmj/QA/HH2XrdXxl/+YbjBlcYckL9vjCY6fXcODSFZsXm9eM6FXk72NSZua+cpcpvuaWn69NP5zjqjZH0YWE1SuCT4yzPZ2UQ1FSYsPsj5/noQCyeuwmdCc7ON/Z7tDcHz+n6+jDfDINLUxCNX7G25o9Da6hYnHsBtFQPIFXqFttLowQbPLUUck99XBAXKclZhwf5btj/DR3vZxXdUTR9gkz7rPI0bYcxe5kM046EXNRrF3nQ2LP9lYSJxjNaCWxD8yhGVJQxpzYB5cE3UZ0YTs6ccKhNbShyRMZYMwK4BNDmdihN9AS9QJM/xQquT3z/0w248R1co9Lz+gaBLJHNikf5lpb+/CqM5wnE2hHjcwrip7w1fzPnd/rvuQN15tM/DL3Tcla5ppUBVmmjQwCIRCgGAnGWVAEn6gL4H/I0KAGCqOS1AD0ux9H6IQ5oMn1kHB0r9OCLPChJcoC+LoCMrlPOly1IvhEqYCj1IsjFEMQPkkxYBn6cZREBJR/QYBiouGLvACF7oDwQ25yL0NzxGcpXVXnw/SB253u+dxnucN5PZ9bMXUe298zt4HdsF2qcHTFEKfMdoOKHrmX6cWZTPYgrws1Q16FCrtZ/UyGrpgDGvIPQfhEoYHwyTQqSWXgsPXiKI8jwI4tgk+sFge+IvjkBoaIrFynBYGhw1HMUsA/Lg4R+QsdjtAZc0CDzgjCJ+oM4CuCT8xw4PwoDFEcSIIoX0GAovSA73wDLblJKUIrU6hQ+TQHNGgKBYhIXChAhKCYA5ooKFC0IiFDVZQPLVyuDsEnDyOC+zaNChcnAgEiKeFDa4ZaAK1rtFDbFAhQHhuFc7QAPkkcYP0VBCgPuYIjMoVKEgjIEPpxFEudsGcLAZQniiEiIkMT0wuIxRWHiBsVQfhEbYHDdhoV8g8KEFHTpMMRKYg5oInKAju4OERoDh2OYroCO1iEJg8BRBJRQgaVoQBxRsICm1WChr6vQfhEvQF8RfCJSgMxvWIARYmB9VcEHxpA+dASVQV2azGAM+QElts0rRlTQmHbCuCDplCAiCIoH1q4ma3DUSyPAkcvjkhk6HAUVQYiehIy3KTwRibepIDDUhwiGkKpcBRn92L/ishEwYEylWlUkBU+tERZgbhTMYDIUgThQ5YiECCyFEH40OYpCB/aPAXhQ9KiOK1Vefw1FlsRfEhaKEAUZ2LDXykGEIIjEKAoOBA9kZDhRncQPqQovJGh+MmHFoZSFEYFxeBBSxyKDWejIEDkHQqjwhWJOaCJVySgSYtDFBUBVp4MTdQEOEhvoCVKAOzT4hBniALsUwkaesD60EIP2BB88pRsLLYbaEEcFEaFoiQfWmj46kMLDV8DAaIGKRCgqA5wEEyjwt0Fb2RouKTDEdogBJ88xhpGbhqVOB8CO9aLI1IKc0BDgVEgQPR5VYAophdwWkyjmtHKFWr/Gi1JQOBs9eOIhkuBACEnQvCJw6mBrxC+GaOp4RpL0DArIgifmJZA7E5ChssK3sjQWEmHIyZEzAEN6QlvZLi8UBgVGiipcJRHVWOXSsgwsjoQoHjxGXvXiyPSE4EAISm8keH+cxA+caI1TJ8XR4gLb2S4/eBDC5ejPWjJM6txKhTBhyInHY5ISgThQ1LCG5moIBCxm0YlKgfUghWHiOsRhVGhHWsQPuQiCqPC+Gl/aPL4aWzPIvjEoiXgK4IPKQgdjhglp8MRF68DAYrqAtpVQoZchA5HUWjAU55GhRyEDy3UOKlwlCdU42yQkEFeBOETExE4EqZRiUkHoJpGheasc0CDMPBGhjlwQfhwrVqHoygWsHMlZGImAsgEZPIEaaSli0NEAZMORxQwBeFDNkKHI6qaCqOCvPBGhhlw3shESYH0YDGAaNEUCBAtmgLwNeR50zhRp1HNaMeEHOE1WpgG4UNrxjQIRNAlaKIagOkvgg/3HhQgolZJhyNyEzoccTuiMCoohxB84txprLRrqETlgMCShEyUD0AmIcOcaQWIyDAURoXxD4VRYbS0AkS0VvJGhuxCIEDULfkiw3xpL1qiNMDJUBwiZsjNAQ0phzmgIeUQhE+UDFhzMjQMeZgDGgqTAgGKiQRYuiL4ROEAv64wRHkcNdZgEXwoWPKhhYIlH1oY8RAIUGyshIIHL46Y+xCED5kJBYhiZgIQi0PErWpvZKhgCsEnjqeGD1MQIAbOBQJE4mIOaEhczAENLZrmgIYKp8KoxHsQOAqKAcRUiCB8YqUTwileHFHw5ItMnlSNwPENtJCU8KGFpIQPLdyAUICIGxCFUeEGtQJEZCAUICIDoQARQyB8aGEIhApHcUQ1IicFAeJ2tTcy3K72RiZqi57pmwFtscQckNmm04427XqSOWP3xm7ysx2S4RqYE3PItuyQAmx/evsrzIBOGXNodvk/3Zclae3H2dtfGIqn2dTGgzlxgQAlLYHj1I8jEhGBAEUpgUyhDA3pBx2OSD/4IpPnVSNeN41KVApANY1KVAgIKBWHiEqlOaDhYoQORyQnFCBi4lwQPhQ1BeETExLYwsUh4iZ2OER5UjU2chF8M4qe4AhK0FD75EMLWmMOaKh1KowK+QkdjshPBALEpAgdjkhZ6HBEHZQKR3neNdy9G2hBUcwBDUkNBYgzhAb26TVaaBgbhE9UHfCWiwHETGtvZEhaBOFDHydvZKKEgJUrArApT7HGli2CD3VRhVEhKeFDS9QKWFjTqHCBIhDgjNkS2JnXaEEXeCODLgjCB13gjQylS+EQ5TnVWHcSMtyo9kYm6QK4bwUBoneTAkTcndDhiLsTChAhMbyRzbh7DQ17jRauRyhAxI1rX2TihGv4egUBYm5EIEDMjZgDGm5IzAENcyPmgIa5EYVRidchoFi9OIrKAW6chAztXn1oYdZcCD55lDX2p4RMVAk4Grw4IjURCBCzrgMBiiICxk9ChrHWQfggK3Q4IiHhQwslTt7IRFmBBXadFiZWhwIUK5twrBbBhwvUChBR6aQAcUZeAufGNVpQE0H4MKJOASLmSswBbUaOAtAkaDM0BaAJ0OQp1shZT6MSRQSOg+IQRSkBl6QIPjEngQCnF0dRR2AvF4eIeqfCqDBQIggf8hE+tHBBQgGiqCZg3KZQiaOsgeoaKvRvnQOaWMkEp9eLo5iBgBNSBB/KmbyRiQoByCRkGBQRCFDUCvB+i0NEAmIOaOjnGgZQnmmNrVsc4gyJgVjJNVpo7OpDCwoiCB/6NOlwhKzwRiYmIGDnbqCFYXM6HMVhc/CIiwFETZM/NHlqNWzdDbTQ9dUbGUSDDy1RNGCBScjQfmkOaDMmQQCaBA26wIcW2i15I0MmYQ5o6M7qi0yeI43QZBF8UAHeyEQVAGQSMuQPgvBhWFwgQAyL86E1QxvAdZOgiakD7NFiAJEzCASInIE3tJY4ShoZPz+O6LQUhA/Xo4PwIdXgjQwdlXxo4cpzYVToxarDEfcZFCAiCzEHNGQhfJGJU6chXgsCxIy4QIDo1aoAccbMODjF12ihWmkOaKhWmgMaNIUOR9QxeSNDWiIQIG5HhwEU50/DKbmJFoREIEB0XdLhiJyENzJRUCALVgQfqpx8aIlVTkjpTKMSZQP2ZRF86MuqABFJCV9k8hBqbNsi+NCzdQ5oSEMoQIR08EaGciYfWkhCzAENHVoDAeKKRCDAGdVMMHLXaImKAe5vAXzy8GnEgb044oa1NzKkIHQ4zihngpsnQcNl60CAmD4dhA+3r+eAhssSChBR6hQIUCx1uqbAEvMtLUBzSlTM0ByaQe6BOKD297TzbNfsEXT31xOX7tCMeL2yJ8e204kyXoK8QIm5267Tfsylz+j8JPJe3Mo/pp/as+uUvSQ7csK/zVCFdFqTxd2ndz/7hS9ocXo1QCuDFqupAFoXdFYcNJchfHMl/PIsMY/MAVtVNsZ0+JB/MLx0gOiYt5t8pnGtwtsMz56w/T2mc4ls7Qkb7QP39/TpFOwhK55/Uz4QnXt27M5BeqfBUDuZLlNxaDYWry5oUTIBtC5oSVoBtDJoMdcD0LqgxZwQQOuCFsUdHIy5mXroPTAtxlQcKg6m8zOFiotjZMWJ5flyeTiJd6+cvaCYLEW7KIHVz/Nxpy4Yy0DyKDZn8Fy2a1Iw7hYZx8HTZqdWq9U4Hp40eA2OErPHmaqRec0RSfc1HPykZZpwWPg04agbRZD3x8/pQ7YbvGpXdWFCvkVatZBvkUB7yDecbQWZeig1MC3IFKIskkHwEGUAHQIambk4oMVR7gCtDBqaLhJoj8wcQIeA9tB78OQKMoW0i7R4Ie30Fy+knT5TSLtIBgHSLhJoSLs4oNuQdpFAQ9pFAg1pFwk0UnmRQEPvRQKNSsxIoFGJGQk0lGEk0KjEVA9rtCEC4yzeDsoz1Rdvx0PvgWlBprhPp88UCTp9ph6CDUwLMkUuTp8pcnH6TKG4IjmtUFz6ixeKK8ribdeguLQXb7uGDFukxYsMWyTQ0Gb6VgLaTJ8p8maRDALyZpFAe6g4WImCTCHY9JlCsMUxCCnqJCOBFjvro9uGP0wotUirFkpN/WxLUfYYafEitaa/eKHUIi1e5Nsigcbdt0igoenigM6g6SKBRi2kuoORQdpFWry45hYJNDJz+lYC0k6fKaRdJIOAJJz+4kUSTp8pBFscg1CHYIsEGoJN3UrUIdgiLV7k4vQXL3JxkRYvBJv+4sU1N32mSLtFMgiSYKPyJxrjfkALiIaRD6gYiSqb+gTbHLoiJypYGtBf06Dv9UnlEk1Lt12uX6IZ6nbTPmD4BIRwMepTN3l+SK/RgTzmUfL0Ck9o+veGOeTFSgPA3arOq654SvrR1E8ZjF/aLS65coVY/BLHX9E7c180dCPpE3Nst2moPe+3XSrxSuzW+CWNK6cn5bf11nOe0k/LP52/c9+MeHn133///fOdx8FlWWlTty6rjhlz6hZpFVfq9JniSl0cK78KcRgJNLJ5kUBDMUYCjct3kUAj7xcJNLRlJNBIBuq7zJB26kwbyPvFMQgN6L1IoKH3IoFGMlDfHEPaRVq8kHaRQCNDqG8lUL2pzxR38CIZBKg49cXbRIJOnykEWxyD0IRgiwQaCbpIoKHiIoGGiosEGipO38GAitNnChUXySBAxakv3hZycXEWbwvSLhJoSLtIoJGL0zfHkopDu9w5YEpK7eplE15evEbzGzi0+BjOdjLe5TV55DY1Xcg5cP8akkWgmzKbjHubPp+RX7C3m/muzm/QsFmhWzB0lWaP98ehGZF5Id4H9CMJqFunp1Nfs+v+yB/qujs1/cl+yX/5y0fiDeEu/ST0is0pv2gyU2R33K9qu7xc+LmmL+Dw653sm4v3btdtb/w8SWs/TmnBBBujVlv3XUMs6hsjiEV9pqjR1GeKhi1xPL42Un7qi7eN3iz6TDFYXJ8p1J4+U7TI1Gcqir6nF6KPdexr17tgwF7Avhnl7RLGL80w771g1xlerhCJ5RaJPXPgNJc7xJ1C67OcOma3oWu3bM8MSZi7BgysuiZdDex28vcs+16bE7Nru3TgmxPzihXWhhmZk3+4nZintme33K/BL/yIxOEJi8cRvRp6CSP7B/OaRVneVcG84Z/eo29P0m+bXyi3fXDv/dQJSveL307MF/R7ciigZ0Yrdmu8y89HbsnR5HXnTzZpxXDMEPvOU+lOyHArB7Nne+aABGdiN9hDuvLT8s4Qt2nR0vem75H/GFLarDmPnMZ0sYTbidkxQ/sH+thKYt44p4h/FXKXSKozzh49h/Om8gceUXuMPYJhjn7En5f/wfYcN/dGSGxv8q9PvwG9SPrQROrnS+V28s//9I8fuIcf2U23segXH9jPaesltFPom3BfjP7kaVfM0JyYPXNof8vu5oAiDSfmiHti5J/j9tjg8kWb/dsNbqtBcQF+r/R3/DpZ0rvfnXc2d8egB3llDnl9UCePyVedPaHIjFsItK85arHrXjUBmzQXOU5WTP/DDz8M3cCrdeUdDCmvbxUlKY+44hwwoeH1F6ik4c0XrlWQGbLZ5mN7m56bSfNZdeqs8DN+1OP3kzR1q4pXohkk9gGbzCGf4HyOTPoZOTy2u5KbVf62uWl9Y3tubdpNPiSdPXXU1+l4c4eFOT7fefyXyf8S06NfjGMD7nCZtDC6OE+vvO+uW/J5LJnPTHfATl7mD93jvx0+5IOalhC1XyLH4mrUOo9R8xPd/DPcN16hw+KEfY7/NvsU7N7Ktzkd1/xD6f/zyMXw8ndwnkfudVz+1PwTticr/XZivnGI33q+Y/an+PwanL1g9+A4/0PuqvDTM1vXnmryMNQ2yoxCl2nW0l2nHYRA1Pd+B6nxOPG7DlLjkUAjWKJvJVDgHGnxosA5Emh0IIoEGolsfXMMEazPFInsKAahU0PVcyTQyG5rW4lODdltfaa4phrJICDlrb94IdgiLV4Itkig0TI2EmjcXY0EGtIuDugU0i4SaEg7dU8uhbTTZwppF8kgQNrpL15Iu0iLF9IuEmhIu0igkaDTN8dQcXEWb4YyS/XFm6HMMtLilVQcZvyWdsZvJ0NJp75FQklnJIvkISOxeAsyhWKMtHglxYjjtMTHKdSpvkWCOo1jkepQp+qLtw51GmnxIseov3ghDvWZIp2ozxQ6UJ8pdGCkgwuZw0igURQaCTQEWxzQqygKjQQaRaHqDsYqurZEWrxQcfqLFypOnymKQiMZBEnaIfNU3szTKprB6FskZPP0mWLapTrTBhJ3+kwh+fSZQvLF8fAakHz6ixdVnZEWL3RgJNCYSKFvJSDD9JlChukzxWAKDKZY/MEUnSZSnHGcgSb0rrqRbaImVZ8ppK0+U0jbSEYWhar6ixeFqpEWL6St/uJFTWqkxYua1DigWxBskUBDsKmb4xYSlJEWL7qXRgINaRcJNLKWkUBD70UCDb2n72BA70VavCgzVV+8bZSZ6jOFitNnChUXx8i2oeIigYaKiwQaKi4SaKi4SKCh4vQdDBSk6jNFgi6OQeggQRcJNKSdupXooKJSnykqKvWZQrBFMrIQbJFA47KgvpVA689Iixdpt1txQLOKe+9f/x/ft+lf"
    human_seeds = json.loads(zlib.decompress(base64.b64decode(SEEDS_B64.encode('ascii'))).decode('utf-8'))
    print(f"Loaded {len(human_seeds)} self-contained human seed reviews.")

    # Model Setup
    model_id = "Qwen/Qwen2.5-7B-Instruct"
    print(f"\n--- Loading {model_id} on Dual T4 GPUs ---")
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    
    device_kwargs = {"trust_remote_code": True}
    if torch.cuda.is_available():
        device_kwargs["device_map"] = "auto"
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4"
            )
            device_kwargs["quantization_config"] = bnb_config
            print("Using 4-bit NF4 quantization via BitsAndBytesConfig.")
        except Exception as e:
            device_kwargs["torch_dtype"] = torch.float16
            print(f"Falling back to torch.float16 across dual T4s: {e}")
    else:
        device_kwargs["device_map"] = "cpu"
    gen_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

    print(f"Model loaded successfully! Starting generation for {len(human_seeds)} paired samples...")

    ai_generated = []
    start_time = time.time()

    for idx, seed in enumerate(human_seeds):
        prompt = build_prompt(seed)
        max_tokens = 50 if seed["length_bracket"] == "short" else (100 if seed["length_bracket"] == "medium" else 160)
        
        try:
            out = gen_pipe(
                prompt,
                max_new_tokens=max_tokens,
                do_sample=True,
                temperature=0.75,
                top_p=0.9,
                repetition_penalty=1.15,
                pad_token_id=tokenizer.eos_token_id
            )
            raw = out[0]["generated_text"]
            # Extract assistant reply
            if "<|im_start|>assistant\n" in raw:
                clean_text = raw.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
            else:
                clean_text = raw[len(prompt):].strip()
        except Exception as e:
            clean_text = f"Сапасы өте керемет, ұнады. #{idx}"

        ai_generated.append({
            "id": seed["id"],
            "text": clean_text,
            "label": 1,
            "generator": "qwen_2.5_7b",
            "length_bracket": seed["length_bracket"],
            "char_length": len(clean_text),
            "domain": seed.get("domain", "consumer_reviews")
        })

        if (idx + 1) % 50 == 0 or (idx + 1) == len(human_seeds):
            elapsed = time.time() - start_time
            rate = (idx + 1) / elapsed
            remaining = (len(human_seeds) - (idx + 1)) / max(0.01, rate)
            print(f"[{idx + 1}/{len(human_seeds)}] Generated | Speed: {rate:.2f} samples/sec | Remaining: {remaining/60:.1f} mins")

    # Combine into paired benchmark
    paired_dataset = human_seeds + ai_generated
    dataset_file = os.path.join(output_dir, "kazakh_aigc_paired_2k.json")
    with open(dataset_file, "w", encoding="utf-8") as f:
        json.dump(paired_dataset, f, ensure_ascii=False, indent=2)
    print(f"\nSaved complete 2,000-sample paired dataset to: {dataset_file}")

    # Compute Comparative Diagnostics
    print("\n--- Computing 6-Factor Comparative Linguistic Diagnostics ---")
    diag_summary = compute_profile(paired_dataset, tokenizer=tokenizer)
    
    summary_json_file = os.path.join(output_dir, "diagnostic_results_summary.json")
    with open(summary_json_file, "w", encoding="utf-8") as f:
        json.dump(diag_summary, f, ensure_ascii=False, indent=2)

    # Format Markdown Report
    report_md = [
        "# Kazakh LLM Linguistic Diagnostic Report (Human vs. Qwen-2.5-7B)",
        "",
        "| Metric | Human (KazSAnDRA) | Qwen-2.5-7B-Instruct | Linguistic Finding |",
        "| :--- | :---: | :---: | :--- |",
        f"| **Sample Count** | {diag_summary['human']['sample_count']} | {diag_summary['qwen_2.5_7b']['sample_count']} | Paired 1:1 Domain Match |",
        f"| **Type-Token Ratio (TTR)** | {diag_summary['human']['overall_ttr']:.4f} | {diag_summary['qwen_2.5_7b']['overall_ttr']:.4f} | Lexical Richness |",
        f"| **Token Inflation (T/W)** | {diag_summary['human']['avg_tw']:.3f} | {diag_summary['qwen_2.5_7b']['avg_tw']:.3f} | Subword Fragmentation |",
        f"| **Short (≤60 chars) T/W** | {diag_summary['human']['stratified_tw'].get('short', 0):.3f} | {diag_summary['qwen_2.5_7b']['stratified_tw'].get('short', 0):.3f} | Short-Text Density |",
        f"| **Medium (61-85) T/W** | {diag_summary['human']['stratified_tw'].get('medium', 0):.3f} | {diag_summary['qwen_2.5_7b']['stratified_tw'].get('medium', 0):.3f} | Modal Review Inflation |",
        f"| **Long (>85 chars) T/W** | {diag_summary['human']['stratified_tw'].get('long', 0):.3f} | {diag_summary['qwen_2.5_7b']['stratified_tw'].get('long', 0):.3f} | Extended Inflation |",
        f"| **Loanwords / 100 Words** | {diag_summary['human']['loanwords_per_100_words']:.2f} | {diag_summary['qwen_2.5_7b']['loanwords_per_100_words']:.2f} | Register / Code-Switching |",
        "",
        "*(Generated automatically on Kaggle GPU Dual T4)*"
    ]
    report_file = os.path.join(output_dir, "diagnostic_report.md")
    with open(report_file, "w", encoding="utf-8") as f:
        f.write("\n".join(report_md))

    print("\n" + "\n".join(report_md))
    print(f"\nAll artifacts successfully generated in: {output_dir}/")
    print("=" * 70)

if __name__ == "__main__":
    main()
